In [1]:
import sys
import os
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath('..')) 
from src.utils.config import USERNAME, ORGNAME
from src.utils.redivis_client import RedivisCatalog, display_hcup_variables # print_redivis_tree
from src.etl.ingest import fetch_hcup, fetch_hcup_ahal
from src.etl.transform import enrich_zip_data, enrich_fips_data

import redivis

# ==========================================
# WARNING: INTERACTIVE AUTHENTICATION
# ==========================================
# If no REDIVIS_API_TOKEN is set in your environment, Redivis will pause 
# execution here and output a URL in the terminal.
# You must click the URL and approve access via your web browser.
# 
# Upon success, Redivis securely caches your credentials locally in:
# Mac/Linux: ~/.redivis/python_credentials
# Windows:   C:\Users\YourUser\.redivis\python_credentials
#
# Future runs on this specific PC will use that hidden cache silently 
# without prompting you again, until the token expires or the folder is deleted.

# Trigger the authentication flow
print(f"Authenticating as `{USERNAME}`...\n")
if redivis.organization(ORGNAME).exists():  #  # <--- This forces to wait for authentication here
    print(f"\033[92m\u2714\033[0m Redivis Python Client `{USERNAME}` successfully authenticated")

Authenticating as `matteo_perini`...

✔ Redivis Python Client `matteo_perini` successfully authenticated


In [2]:
#hcup_cu=RedivisCatalog(ORGNAME)

#vars_df=display_hcup_variables(catalog = hcup_cu, state= 'New York', year= 2017, db_type= 'sedd')

#display(vars_df['Variable'])
#display(hcup_cu.datasets)

#display(hcup_cu.get_tables("kentucky_hcup:02jh:v3_1"))

#display(hcup_cu.get_variables("kentucky_hcup:02jh:v3_1","ky_sedd_2017_ahal:1hxt"))


In [3]:
#print_redivis_tree(ORGNAME, limit_tables=0) # default: limit_tables=3

In [4]:
#SEDD 

selected_cntys = {
    'Arizona': ('Pima, AZ', 'Pinal, AZ', 'Yavapai, AZ', 'Coconino, AZ'),  
    'Iowa': ('Polk, IA', 'Linn, IA', 'Scott, IA', 'Johnson, IA', 'Black Hawk, IA'), 
    'Kentucky': ('Jefferson, KY', 'Fayette, KY', 'Kenton, KY', 'Boone, KY', 'Warren, KY'), 
    'New Jersey': ('Bergen, NJ', 'Middlesex, NJ', 'Essex, NJ', 'Hudson, NJ', 'Monmouth, NJ'), 
    'New York': ('New York, NY', 'Kings, NY', 'Queens, NY', 'Bronx, NY', 'Richmond, NY'), 
    'North Carolina': ('Wake, NC', 'Mecklenburg, NC', 'Guilford, NC', 'Forsyth, NC', 'Cumberland, NC'), 
    'Utah': ('Salt Lake, UT', 'Utah, UT', 'Davis, UT', 'Weber, UT', 'Washington, UT')
}


#selected_cntys = {'Iowa': ('Polk, IA', 'Linn, IA', 'Scott, IA', 'Johnson, IA', 'Black Hawk, IA')}


hcup_cu=RedivisCatalog(ORGNAME)
db_type="sedd"


cols_to_pull_icd9 = ["KEY", "AGE", "ZIP", "AMONTH", "AYEAR", "DSHOSPID", "HOSPID"]
cols_to_pull_icd10 = ["KEY", "ZIP", "AMONTH", "AYEAR", "DSHOSPID", "HOSPID"]

# ICD19
years_icd9=[str(y) for y in range(2006, 2015)] + ["2015q1q3"]
prefix_icd9="ECODE"
vals_icd9=[f"E{i}" for i in range(950, 960)]

# ICD10
years_icd10=["2015q4"] + [str(y) for y in range(2016, 2021)]
prefix_icd10="I10_DX"
# 1. Suicidal Ideation and Unspecified Attempts (Exact matches)
# R45851: Suicidal ideation
# T1491xx: Suicide attempt (Initial, Subsequent, Sequela)
vals_ideation_attempt = ['R45851', 'T1491XA', 'T1491XD', 'T1491XS']

# 2. Intentional Self-Harm by mechanism (Base codes X71 through X83)
vals_self_harm_x = [f"X{i}" for i in range(71, 84)]
# 3. Intentional Self-Poisoning (Base codes T36 through T50)
vals_poisoning_t = [f"T{i}" for i in range(36, 51)]

# Combine them all into one master tuple of prefixes
# (Tuples are required withpandas .str.startswith())
vals_icd10 = tuple(vals_ideation_attempt + vals_self_harm_x + vals_poisoning_t)


In [5]:
sedd_data = {}

for state in selected_cntys.keys():
    print(f"--- Processing {state} ---")
# 1. Fetch Core Data
    df_icd9 = fetch_hcup(hcup_cu, state, db_type, ["2015q1q3"], cols_to_pull_icd9, prefix_icd9, vals_icd9, return_icd_cols=False)
    df_icd10 = fetch_hcup(hcup_cu, state, db_type, ["2015q4"], cols_to_pull_icd10, prefix_icd10, vals_icd10, return_icd_cols=False)
    
    # 2. Extract KEYs to pass to the AHAL fetcher
    keys = []
    if 'KEY' in df_icd9.columns: keys += df_icd9['KEY'].dropna().astype(str).tolist()
    if 'KEY' in df_icd10.columns: keys += df_icd10['KEY'].dropna().astype(str).tolist()
    
    df_ahal = fetch_hcup_ahal(hcup_cu, state, ["2015q1q3", "2015q4"], core_keys=list(set(keys)))

# 3. Clean strings and handle HCUP negative missing codes
    hcup_missing_codes = [-9999, -9998, -99, -9, '-9999', '-9998', '-99', '-9']
    for df in [df_icd9, df_icd10, df_ahal]:
        df.replace(hcup_missing_codes, np.nan, inplace=True)
        for col in ['DSHOSPID', 'HOSPID', 'AYEAR', 'YEAR', 'KEY']:
            if col in df.columns:
                mask = df[col].notna()
                df.loc[mask, col] = (df.loc[mask, col]
                                     .astype(str)
                                     .str.replace(r'\.0$', '', regex=True)
                                     .str.strip()
                                     .str.replace(r'^0+(?!$)', '', regex=True))
                df[col] = df[col].astype(str)

    # 4. Smart Merge Execution
    def smart_merge_ahal(df_core, df_ahal):
        if df_core.empty or df_ahal.empty:
            return df_core
            
        # A. The Iowa Scenario: Both have KEY
        if 'KEY' in df_core.columns and 'KEY' in df_ahal.columns:
            overlap = [c for c in df_ahal.columns if c in df_core.columns and c != 'KEY']
            ahal_clean = df_ahal.drop(columns=overlap, errors='ignore')
            return df_core.merge(ahal_clean, on='KEY', how='left')
            
        # B. Standard Scenario: Merge on Hospital IDs
        if 'AYEAR' in df_core.columns:
            df_core = df_core.rename(columns={'AYEAR': 'YEAR'})
            
        has_hospid = 'HOSPID' in df_core.columns and df_core['HOSPID'].notna().sum() > 0
        has_dshospid = 'DSHOSPID' in df_core.columns and df_core['DSHOSPID'].notna().sum() > 0
        
        if has_hospid:
            merge_key, drop_key = 'HOSPID', 'DSHOSPID'
        elif has_dshospid:
            merge_key, drop_key = 'DSHOSPID', 'HOSPID'
        else:
            df_core['HFIPSSTCO'] = np.nan
            return df_core 
            
        ahal_clean = df_ahal.drop(columns=[drop_key], errors='ignore')
        df_core[merge_key] = df_core[merge_key].astype(str)
        ahal_clean[merge_key] = ahal_clean[merge_key].astype(str)
        
        return df_core.merge(ahal_clean, on=[merge_key, 'YEAR'], how='left')

    df_icd9 = smart_merge_ahal(df_icd9, df_ahal)
    df_icd10 = smart_merge_ahal(df_icd10, df_ahal)

    # 5. Enrich with FIPS Geodata
    df_icd9 = enrich_fips_data(df_icd9)
    df_icd10 = enrich_fips_data(df_icd10)

    # 6. Store
    sedd_data[state] = pd.concat([df_icd9, df_icd10], ignore_index=True)

--- Processing Arizona ---
Fetching Arizona sedd 2015q1q3...


  0%|          | 0/6964 [00:00<?, ?it/s]

Fetching Arizona sedd 2015q4...


  0%|          | 0/13199 [00:00<?, ?it/s]

  0%|          | 0/113 [00:00<?, ?it/s]

  0%|          | 0/113 [00:00<?, ?it/s]

  0%|          | 0/77 [00:00<?, ?it/s]

  0%|          | 0/77 [00:00<?, ?it/s]

ValueError: You are trying to merge on float64 and object columns for key 'HFIPSSTCO'. If you wish to proceed you should use pd.concat

the script still messes up when trying to merge ahal to get the hospital fips (and then the hospital county name) in the final df.

i give up now and focus on trying to find "clusters" on the nvdrs data at county level / age groups

In [ ]:

# Test the output for New York
ia_df = sedd_data['Iowa']